# --------------------------------------------------
#  -> Create a Chatbot 
#  -> The output of Chatbot in Console(cmd) 
# ----------------------------------------------------

### Version 1: 

In [ ]:


# import necessary libraries
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# load the openai api key from .env file
load_dotenv()

# instance of openai model
model = ChatOpenAI(
    model = "gpt-4o"
)

while True:
    user_input = input("You: ")

    if user_input == "exit":
        break

    response = model.invoke(user_input)

    print("AI: ", response.content)

- Key Problem:
    - This Chatbot has no previous content 
    - This chatbot(version 1) can't have the previous conversational history

- Example input:
    - user query 1  : Hi
    - AI  response 1: Hello! How can I assist you today?

    - user query 2  : which on is greater 2 or 1
    - AI  response 2: 2 is greater then 1



    - user query 3  : Then multiply the bigger number with 10
    - AI  response 3: Let's say the bigger number is x.
                    - Multiplying x by 10 we get: 10x

    - so, here, the problem, Chatbot has no previous content.

### Version 2
- introduce memory

In [ ]:


# import necessary libraries
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# load the openai api key from .env file
load_dotenv()

# instance of openai model
model = ChatOpenAI(
    model = "gpt-4o"
)

# memory
chat_history = []

while True:
    user_input = input("You: ")
    chat_history.append(user_input)

    if user_input == "exit":
        break

    response = model.invoke(chat_history)
    # store the AI chat response
    chat_history.append(response.content)

    print("AI: ", response.content)

# the entire chat history
print(chat_history)

- Key Problem:
    - This Chatbot has no previous content 
    - This chatbot(version 1) can't have the previous conversational history

- Example input:
    - user query 1  : Hi
    - AI  response 1: Hello! How can I assist you today?

    - user query 2  : which on is greater 2 or 1
    - AI  response 2: 2 is greater then 1



    - user query 3  : Then multiply the bigger number with 10
    - AI  response 3: 2 multiply by 10 is equal to 10.

    - so, here, the problem, There is not sequence what is the user query and the AI response because we store all message but can't define the which one from User query and which one come from AI response 

### Version 3: -- using Langchain
- Store the history into correct format
- store the message with labeling 
    - label with user_message
    - label with AI response message 

    - This formation is provide by the langchain

### LangChain has three message
- 1. System message

- 2. Human Message

- 3. AI Message


- The gold standard for modern LangChain apps. Uses message roles to structure conversations exactly how LLMs are trained.

In [ ]:
""" 
- The Three Core Roles:
┌─────────────────────────────────────────────────────┐
│                    MESSAGE ROLES                     │
├──────────┬──────────────────────────────────────────┤
│ 🎭 system│ Sets the AI's persona, rules, and        │
│          │ behavior. The "job description."          │
│          │ → "You are a senior data analyst."        │
├──────────┼──────────────────────────────────────────┤
│ 👤 human │ The user's actual question or input.      │
│          │ → "Analyze this sales data."              │
├──────────┼──────────────────────────────────────────┤
│ 🤖 ai    │ The AI's previous response (for multi-   │
│          │ turn context).                            │
│          │ → "Based on the data, revenue grew 15%."  │
└──────────┴──────────────────────────────────────────┘


"""

In [ ]:
# Version 3: using langChain
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from dotenv import load_dotenv

load_dotenv()


# instance of ChatOpenAI model
model = ChatOpenAI(
    model = "gpt-4o"
)

# define the  messages (message memory)
messages = [
    SystemMessage(context = "You are a helpful assistant"),
    HumanMessage(content = "Tell me about the langchain")

]

# invoke function
response = model.invoke(messages)

# store the AI response into message memory
messages.append(AIMessage(response.content))

# Display all conversation (all message memory)
print(messages)

### Key Takeaway Prompt(Message) in LangChain:
- Message has two type
    - 1. single message(single turn stand along queries)
        - i. static message

        - ii. Dynamic message
            - langchain class: PromptTemplate
    
    - 2. list of message (multi-turn conversation )
        - i. static message
            - SystemMessage, HumanMessage, AIMessage
        
        - ii. Dynamic message
            - langchain class: ChatPromptTemplate


- where, 
    - System → defines the model's behavior
    - Human  → provides the user's request
    - AI     → represents previous assistant responses


In [ ]:
"""  
System:
You are an experienced Python instructor.

Human:
Explain decorators.

AI:
A decorator is...

Human:
Give me a real-world example.

"""

### ChatPromptTemplate

- ChatPromptTemplate: 
    - ChatPromptTemplate is a LangChain class used to create reusable, structured chat prompts.

- use: 
    - when we work with list of messages with dynamic messages


    

In [3]:
from langchain_core.prompts import ChatPromptTemplate

# Create a chat prompt template
prompt = ChatPromptTemplate.from_messages([
    # Define the model's role
    ("system", "You are an experienced {domain} instructor."),

    # Define the user's request
    ("human", "Explain {topic} in simple language.")
])

formatted_prompt = prompt.invoke({
    "domain": "AI",
    "topic": "Generative AI"
})

print(formatted_prompt)

messages=[SystemMessage(content='You are an experienced AI instructor.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain Generative AI in simple language.', additional_kwargs={}, response_metadata={})]


### Message Placeholder

- Definition:
    - MessagesPlaceholder is a LangChain component that allows us to create a placeholder for a dynamic list of chat messages.

- In simple terms:
    - MessagesPlaceholder says: "At this position, insert whatever messages I provide later."



In [ ]:
"""   
System message
      ↓
"You're a helpful assistant."

MessagesPlaceholder
      ↓
[Previous conversation goes here]

Human message
      ↓
"Answer my latest question."

"""

#### Why do we need it?
- We need a way to dynamically inject the conversation. then the MessagesPlaceholder provides that mechanism.

In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage


# ---------------------------------------------------------
# Step 1: Create the chat prompt template
# ---------------------------------------------------------

prompt = ChatPromptTemplate.from_messages([

    # System message defines the assistant's behavior
    ("system", "You are a helpful AI instructor."),

    # This is the dynamic injection point.
    # A list of previous messages will be inserted here.
    MessagesPlaceholder(variable_name="chat_history"),

    # Current user question
    ("human", "{question}")
])


# ---------------------------------------------------------
# Step 2: Create conversation history
# ---------------------------------------------------------

chat_history = [

    HumanMessage(
        content="What is Machine Learning?"
    ),

    AIMessage(
        content="Machine Learning is a subset of AI that allows "
                "computers to learn patterns from data."
    ),

    HumanMessage(
        content="Where is it used?"
    ),

    AIMessage(
        content="It is used in recommendation systems, fraud detection, "
                "medical diagnosis, and many other applications."
    )
]


# ---------------------------------------------------------
# Step 3: Provide dynamic values to the prompt
# ---------------------------------------------------------

formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "question": "Give me a simple real-world example."
})


# ---------------------------------------------------------
# Step 4: Display the resulting messages
# ---------------------------------------------------------

for message in formatted_prompt.messages:
    print(f"{message.type}: {message.content}")

system: You are a helpful AI instructor.
human: What is Machine Learning?
ai: Machine Learning is a subset of AI that allows computers to learn patterns from data.
human: Where is it used?
ai: It is used in recommendation systems, fraud detection, medical diagnosis, and many other applications.
human: Give me a simple real-world example.


#### MessagesPlaceholder with a Model

In [ ]:
# MessagesPlaceholder with a Model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv


# Load environment variables from .env
load_dotenv()


# Create the chat model
model = ChatOpenAI(
    model="gpt-4o"
)


# Create the prompt
prompt = ChatPromptTemplate.from_messages([

    # Define the assistant's role
    ("system", "You are a helpful AI tutor."),

    # Dynamically insert previous conversation
    MessagesPlaceholder(variable_name="chat_history"),

    # Add the current user question
    ("human", "{question}")
])


# Previous conversation
chat_history = [
    HumanMessage(
        content="What is Deep Learning?"
    ),

    AIMessage(
        content="Deep Learning is a branch of Machine Learning "
                "that uses neural networks with multiple layers."
    )
]


# Build the complete prompt
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "question": "How is it different from traditional Machine Learning?"
})


# Send the formatted prompt to the model
response = model.invoke(formatted_prompt)


# Print only the model's generated text
print(response.content)

In [ ]:
## The Completed Summary of Placeholder

"""    
┌──────────────────────────────────────────────────────────────┐
│                  MessagesPlaceholder                          │
│                                                              │
│  WHAT:  A dynamic slot that expands into N messages          │
│  WHY:   Chat history is variable-length; can't hardcode      │
│  WHERE: Inside ChatPromptTemplate.from_messages([...])       │
│                                                              │
│  SYNTAX:                                                     │
│    MessagesPlaceholder(variable_name="chat_history")         │
│                                                              │
│  INPUT:  List[BaseMessage] (HumanMessage, AIMessage, etc.)   │
│  LENGTH: 0 to ∞ (but trim in practice!)                      │
│                                                              │
│  USE CASES:                                                  │
│    ├── Chatbots with memory                                  │
│    ├── RAG with conversation context                         │
│    ├── Tool-calling agents (AIMessage + ToolMessage)         │
│    ├── Multi-turn code assistants                            │
│    ├── Multi-agent conversations                             │
│    └── LangGraph state management                            │
│                                                              │
│  GOLDEN RULE:                                                │
│  "If your app has conversations, you NEED MessagesPlaceholder"│
└──────────────────────────────────────────────────────────────┘



"""